# DS-GA 1019

# Lab 6 

## Part 2

## Feb. 26, 2026

## Numba

### What is Numba?

a JIT compiler for Python that:
- generates optimized machine code using LLVM compiler
- integrates well with the Scientific Python stack (Numpy) 

<img src="https://i.ibb.co/C2qPRJY/img1.png" width="400" height="400">

Interpretation:
1. Compile to bytecode
2. Interpret in a virtual machine (e.g. Python, Java)
    
Ahead of time compilation:
 1. Compile to binary code
 2. Execute on hardware (e.g. C, C++)
    

   ### Just in time compilation
<img src="https://i.ibb.co/nmtzCfP/img2.png" width="400" height="400">


#### Array sum
The function below is a naive sum function

In [1]:
import numpy as np

def sum_array(arr):
    r, c = arr.shape

    mysum = 0
    for i in range(r):
        for j in range(c):
            mysum += arr[i, j]

    return mysum

In [2]:
arr = np.random.random((300, 300))
sum_array(arr)

np.float64(45004.23572546394)

In [3]:
plain = %timeit -o sum_array(arr)

7.28 ms ± 113 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### As a function call

In [4]:
from numba import jit

sum_array_numba = jit()(sum_array)

In [5]:
jitted = %timeit -o sum_array_numba(arr)

86.2 μs ± 3.3 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
# speed boost
plain.best / jitted.best

85.74110966748597

### As a decorator

In [7]:
@jit
def sum_array(inp):
    r, c = arr.shape

    mysum = 0
    for i in range(r):
        for j in range(c):
            mysum += arr[i, j]

    return mysum


In [8]:
sum_array(arr)

45004.23572546394

In [9]:
%timeit sum_array(arr)

85.1 μs ± 274 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Defining ufuncs using vectorize

In [10]:
import math

def f(a, b):
    return math.sin(a**2) * math.exp(b)


In [11]:
f(1,1)

2.2873552871788423

In [12]:
a = np.ones((5,5))
b = np.ones((5,5))

In [13]:
from numba import vectorize

In [14]:
vec_f = vectorize()(f)

vec_f(a,b)

array([[2.28735529, 2.28735529, 2.28735529, 2.28735529, 2.28735529],
       [2.28735529, 2.28735529, 2.28735529, 2.28735529, 2.28735529],
       [2.28735529, 2.28735529, 2.28735529, 2.28735529, 2.28735529],
       [2.28735529, 2.28735529, 2.28735529, 2.28735529, 2.28735529],
       [2.28735529, 2.28735529, 2.28735529, 2.28735529, 2.28735529]])

#### How does it compare to just using NumPy? 

In [15]:
def numpy_f(a, b):
    return np.sin(a**2) * np.exp(b)


In [16]:
a = np.random.random((1000, 1000))
b = np.random.random((1000, 1000))

In [17]:
%timeit vec_f(a, b)

6.27 ms ± 118 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
%timeit numpy_f(a, b)

7.83 ms ± 140 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


What happens if we specify a signature?

In [19]:
vec_f = vectorize('float64(float64, float64)')(f)

%timeit vec_f(a, b)

6.09 ms ± 23.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# fastest with target='parallel'
vec_f = vectorize('float64(float64, float64)', target='parallel')(f)

%timeit vec_f(a, b)

1.04 ms ± 11.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
